In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os

from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader

In [18]:
data = pd.read_csv("jm1.csv")

data.head()

,loc,v(g),ev(g),iv(g),n,v,l,d,i,e,...,lOCode,lOComment,lOBlank,locCodeAndComment,uniq_Op,uniq_Opnd,total_Op,total_Opnd,branchCount,defects
0,1.1,1.4,1.4,1.4,1.3,1.30,1.30,1.30,1.30,1.30,...,2,2,2,2,1.2,1.2,1.2,1.2,1.4,False
1,1.0,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.00,1.00,...,1,1,1,1,1.0,1.0,1.0,1.0,1.0,True
2,72.0,7.0,1.0,6.0,198.0,1134.13,0.05,20.31,55.85,23029.10,...,51,10,8,1,17.0,36.0,112.0,86.0,13.0,True
3,190.0,3.0,1.0,3.0,600.0,4348.76,0.06,17.06,254.87,74202.67,...,129,29,28,2,17.0,135.0,329.0,271.0,5.0,True
4,37.0,4.0,1.0,4.0,126.0,599.12,0.06,17.19,34.86,10297.30,...,28,1,6,0,11.0,16.0,76.0,50.0,7.0,True


In [3]:
data = data.drop_duplicates()

print("Dataset shape after removing duplicates:", data.shape)

Dataset shape after removing duplicates: (8908, 22)


In [4]:
print(data.isnull().sum())
data = data.fillna(data.median())

loc                  0
v(g)                 0
ev(g)                0
iv(g)                0
n                    0
v                    0
l                    0
d                    0
i                    0
e                    0
b                    0
t                    0
lOCode               0
lOComment            0
lOBlank              0
locCodeAndComment    0
uniq_Op              0
uniq_Opnd            0
total_Op             0
total_Opnd           0
branchCount          0
defects              0
dtype: int64


In [6]:
data['defects'] = data['defects'].apply(lambda x: 1 if x > 0 else 0)  #converting defects -> 0, if not 1

In [29]:
# label column
y = data['defects']

# remove label column
X = data.drop(columns=['defects','branchCount'])

In [30]:
print(X.shape)

(13204, 20)


In [31]:
scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

In [32]:
# To handle class imbalance
smote = SMOTE(random_state=42)

X_balanced, y_balanced = smote.fit_resample(X_scaled, y)

In [33]:
X = np.array(X_balanced)
y = np.array(y_balanced)

In [34]:
#Converting to images 
def metrics_to_image(row):

    img = row.reshape(4,5)

    img = (img - img.min()) / (img.max() - img.min())

    img = (img * 255).astype(np.uint8)

    return img


In [35]:
images = []

for row in X:
    images.append(metrics_to_image(row))

images = np.array(images)

In [37]:
#padding and resizing 
def pad_resize(img):

    padded = np.pad(img, pad_width=20, mode='symmetric')

    resized = cv2.resize(padded,(224,224))

    return resized

processed_images = []

for img in images:
    processed_images.append(pad_resize(img))

processed_images = np.array(processed_images)

In [39]:
#train-test splitting data
X_train, X_test, y_train, y_test = train_test_split(
    processed_images,
    y,
    test_size=0.2,
    random_state=42
)

In [41]:
#creating pytorch dataset
class MetricImageDataset(Dataset):

    def __init__(self, images, labels, transform=None):

        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img = Image.fromarray(self.images[idx]).convert("RGB")

        if self.transform:
            img = self.transform(img)

        label = self.labels[idx]

        return img, label
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = MetricImageDataset(X_train,y_train,transform)
test_dataset = MetricImageDataset(X_test,y_test,transform)

train_loader = DataLoader(train_dataset,batch_size=16,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=16)

In [42]:
#loading pre-trained GoogleNet
model = models.googlenet(pretrained=True)

C:\Users\avina\PythonPortable\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\avina\PythonPortable\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=GoogLeNet_Weights.IMAGENET1K_V1`. You can also use `weights=GoogLeNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/googlenet-1378be20.pth" to C:\Users\avina/.cache\torch\hub\checkpoints\googlenet-1378be20.pth


100%|█████████████████████████████████████████████████████████████████████████████| 49.7M/49.7M [00:00<00:00, 58.5MB/s]


In [43]:
#freezing layers 
for param in model.parameters():
    param.requires_grad = False

#Removing final layer
feature_extractor = nn.Sequential(*list(model.children())[:-1])

feature_extractor.eval()

Sequential(
  (0): BasicConv2d(
    (conv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (2): BasicConv2d(
    (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (3): BasicConv2d(
    (conv): Conv2d(64, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(192, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (4): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (5): Inception(
    (branch1): BasicConv2d(
      (conv): Conv2d(192, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    )
    (bra

In [44]:
# Extracting features 

def extract_features(loader):

    features = []
    labels = []

    with torch.no_grad():

        for imgs, lbls in loader:

            outputs = feature_extractor(imgs)

            outputs = outputs.view(outputs.size(0), -1)

            features.append(outputs.numpy())
            labels.append(lbls.numpy())

    features = np.vstack(features)
    labels = np.hstack(labels)

    return features, labels

In [45]:
train_features, train_labels = extract_features(train_loader)


test_features, test_labels = extract_features(test_loader)

In [46]:
#training Random Forest

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(train_features, train_labels)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [48]:
#predictions

predictions = rf.predict(test_features)
print(predictions)

[ True  True  True ... False False False]


In [49]:
#Evaluation of model

f1 = f1_score(test_labels, predictions)

print("F1 Score:", f1)

print(classification_report(test_labels, predictions))

F1 Score: 0.8695457596777803
              precision    recall  f1-score   support

       False       0.89      0.84      0.87      2267
        True       0.85      0.89      0.87      2174

    accuracy                           0.87      4441
   macro avg       0.87      0.87      0.87      4441
weighted avg       0.87      0.87      0.87      4441

